<a href="https://colab.research.google.com/github/yklegend/AI-PROJECTS/blob/main/Satteliteimage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahmoudreda55/satellite-image-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'satellite-image-classification' dataset.
Path to dataset files: /kaggle/input/satellite-image-classification


In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        break

/kaggle/input/satellite-image-classification/data/cloudy/train_17406.jpg
/kaggle/input/satellite-image-classification/data/desert/desert(14).jpg
/kaggle/input/satellite-image-classification/data/green_area/Forest_2838.jpg
/kaggle/input/satellite-image-classification/data/water/SeaLake_926.jpg


In [ ]:
import torch
from torch import nn


In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader

# Define transformations for the images
transform = transforms.Compose([
    transforms.Resize((28, 28)),  # Resize images to 224x224 pixels
    transforms.ToTensor(),         # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize image data
])

# Load the dataset using ImageFolder
dataset = torchvision.datasets.ImageFolder(root=f"{path}/data", transform=transform)

# Get the number of classes and class names
num_classes = len(dataset.classes)
class_names = dataset.classes
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")

# Define the split ratio (e.g., 80% for training, 20% for testing)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Perform the train-test split
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(f"Training dataset size: {len(train_dataset)}")
print(f"Testing dataset size: {len(test_dataset)}")

# Create data loaders for training and testing
batch_size = 32 # You can adjust this based on your GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of batches in training loader: {len(train_loader)}")
print(f"Number of batches in testing loader: {len(test_loader)}")

Number of classes: 4
Class names: ['cloudy', 'desert', 'green_area', 'water']
Training dataset size: 4504
Testing dataset size: 1127
Number of batches in training loader: 141
Number of batches in testing loader: 36


In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=4):
        super(CNN, self).__init__()
        # First Convolutional Block
        # Input: 3 channels (RGB image), size 28x28
        # nn.Conv2d(in_channels, out_channels, kernel_size, padding)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU() # Activation function
        # nn.MaxPool2d(kernel_size, stride)
        # Output after pooling: 16 channels, size 14x14 (28 / 2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Second Convolutional Block
        # Input: 16 channels, size 14x14
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU() # Activation function
        # Output after pooling: 32 channels, size 7x7 (14 / 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layer (Classifier)
        # The input to the FC layer needs to be flattened from 3D (channels, height, width) to 1D
        # Calculate input features: channels * height * width = 32 * 7 * 7
        self.fc = nn.Linear(32 * 7 * 7, num_classes)

    def forward(self, x):
        # Apply first conv, relu, and pool
        x = self.pool1(self.relu1(self.conv1(x)))
        # Apply second conv, relu, and pool
        x = self.pool2(self.relu2(self.conv2(x)))
        # Flatten the tensor for the fully connected layer
        # -1 infers the batch size automatically
        x = x.view(-1, 32 * 7 * 7)
        # Apply fully connected layer
        x = self.fc(x)
        return x

# Instantiate the model with the number of classes identified earlier
model = CNN(num_classes=num_classes)
print(model)

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=1568, out_features=4, bias=True)
)


In [ ]:
import torch.optim as optim

# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001) # You can adjust the learning rate (lr)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [ ]:
num_epochs = 10 # You can adjust the number of epochs

# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using device: {device}")

for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_accuracy = 100 * correct_train / total_train
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Train Accuracy: {train_accuracy:.2f}%')

    # Evaluation on the test set
    model.eval() # Set the model to evaluation mode
    correct_test = 0
    total_test = 0
    with torch.no_grad(): # Disable gradient calculation during evaluation
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_accuracy = 100 * correct_test / total_test
    print(f'Test Accuracy: {test_accuracy:.2f}%')

print('Finished Training')


Using device: cuda
Epoch 1/10, Loss: 0.5044, Train Accuracy: 76.31%
Test Accuracy: 77.28%
Epoch 2/10, Loss: 0.3762, Train Accuracy: 83.13%
Test Accuracy: 87.05%
Epoch 3/10, Loss: 0.3104, Train Accuracy: 87.19%
Test Accuracy: 88.82%
Epoch 4/10, Loss: 0.2743, Train Accuracy: 88.94%
Test Accuracy: 88.38%
Epoch 5/10, Loss: 0.2397, Train Accuracy: 90.63%
Test Accuracy: 91.57%


# Task
Save the trained PyTorch model and create a Gradio application to predict image classes.

## Save Trained Model

### Subtask:
Save the trained PyTorch model's state dictionary to a file (e.g., `model.pth`) so it can be loaded later for inference.


**Reasoning**:
Save the trained model's state dictionary to a file using `torch.save()`.



In [ ]:
torch.save(model.state_dict(), 'model.pth')
print("Model saved to model.pth")

**Reasoning**:
The previous code failed because the `torch` library was not explicitly imported in the cell where `torch.save()` was called, leading to a `NameError`. I will add `import torch` to the cell to ensure it is defined before use.



In [ ]:
import torch
torch.save(model.state_dict(), 'model.pth')
print("Model saved to model.pth")

### Subtask:
Create a Gradio application to predict image classes using the trained PyTorch model.

#### Instructions:
1. Define a prediction function that takes an image as input and returns predicted class probabilities and class names.
2. Create a Gradio interface using `gr.Interface` with the prediction function, appropriate input and output components, and example images.
3. Launch the Gradio application.

**Reasoning**:
Before proceeding with creating the Gradio application, it is essential to ensure that the `gradio` library is installed in the environment. This step will install the necessary package.



In [ ]:
pip install gradio

**Reasoning**:
Now that Gradio is installed, I will define the prediction function `predict_image` for the Gradio interface. This function will load the saved model, apply the same transformations used during training to the input image, perform inference, and return the predicted class probabilities and class names. I also need to import `gradio`, `Image` from `PIL`, and `torch.nn.functional` for softmax.



In [ ]:
import gradio as gr
from PIL import Image
import torch.nn.functional as F

# Load the model with its state dictionary
model = CNN(num_classes=num_classes) # Re-instantiate the model structure
model.load_state_dict(torch.load('model.pth')) # Load the saved weights
model.eval() # Set to evaluation mode

def predict_image(image):
    # Ensure the image is in RGB format
    if image.mode != 'RGB':
        image = image.convert('RGB')

    # Apply the same transformations as during training
    input_tensor = transform(image)
    input_batch = input_tensor.unsqueeze(0)  # Add a batch dimension

    # Move the input to the same device as the model
    input_batch = input_batch.to(device)
    model.to(device)

    with torch.no_grad():
        output = model(input_batch)

    # Get probabilities
    probabilities = F.softmax(output, dim=1)[0]

    # Convert to a dictionary for Gradio output
    confidences = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}

    return confidences

print("Prediction function defined.")

**Reasoning**:
Now that the prediction function is defined, I will create the Gradio interface. This involves using `gr.Interface` with the `predict_image` function, setting the input component to `gr.Image()`, the output to `gr.Label()` to display probabilities, and providing example image paths for demonstration. Finally, I will launch the interface.



In [ ]:
import gradio as gr

# Get example image paths from the dataset for Gradio examples
example_images = []

# Take one example from each class for demonstration
# The structure is /kaggle/input/satellite-image-classification/data/{class_name}/{image_name}

# Example for 'cloudy'
example_images.append(os.path.join(path, 'data', 'cloudy', 'cloudy_1.jpg'))
# Example for 'desert'
example_images.append(os.path.join(path, 'data', 'desert', 'desert(1).jpg'))
# Example for 'green_area'
example_images.append(os.path.join(path, 'data', 'green_area', 'Forest_1.jpg'))
# Example for 'water'
example_images.append(os.path.join(path, 'data', 'water', 'SeaLake_1.jpg'))


# Create the Gradio interface
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(),
    title="Satellite Image Classifier",
    description="Upload a satellite image (cloudy, desert, green area, or water) and get its predicted class.",
    examples=example_images
)

# Launch the Gradio app
interface.launch(debug=True)
print("Gradio interface launched.")

# Task
Save the entire trained PyTorch model object as `model.pkl`, then define a prediction function that loads this `model.pkl` for inference, and finally create and launch a Gradio application with this updated prediction function to classify satellite images.

## Save Full Trained Model as PKL

### Subtask:
Save the entire trained PyTorch model object (including its architecture and learned parameters) to a file named `model.pkl` using `torch.save()`.


**Reasoning**:
Save the entire trained PyTorch model object, including its architecture, to a file named `model.pkl`.



In [ ]:
torch.save(model, 'model.pkl')
print("Entire model saved to model.pkl")

**Reasoning**:
The previous steps successfully saved the entire model object to `model.pkl`. Now, I need to define a prediction function for the Gradio interface that loads this `model.pkl`, applies the necessary transformations to the input image, performs inference, and returns the predicted class probabilities and class names. This will ensure that the Gradio app uses the fully saved model.



In [ ]:
import gradio as gr
from PIL import Image
import torch.nn.functional as F

# Load the entire model from the .pkl file
model_pkl = torch.load('model.pkl')
model_pkl.eval() # Set to evaluation mode

def predict_image_pkl(image):
    # Ensure the image is in RGB format
    if image.mode != 'RGB':
        image = image.convert('RGB')

    # Apply the same transformations as during training
    input_tensor = transform(image)
    input_batch = input_tensor.unsqueeze(0)  # Add a batch dimension

    # Move the input to the same device as the model
    input_batch = input_batch.to(device)
    model_pkl.to(device)

    with torch.no_grad():
        output = model_pkl(input_batch)

    # Get probabilities
    probabilities = F.softmax(output, dim=1)[0]

    # Convert to a dictionary for Gradio output
    confidences = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}

    return confidences

print("Prediction function using model.pkl defined.")

**Reasoning**:
The previous code failed to load the model due to an `UnpicklingError` related to `weights_only` being set to `True` by default in newer PyTorch versions. To fix this, I will explicitly set `weights_only=False` in `torch.load()` to allow the full model object, including its class definition, to be loaded.



In [ ]:
import gradio as gr
from PIL import Image
import torch.nn.functional as F

# Load the entire model from the .pkl file
model_pkl = torch.load('model.pkl', weights_only=False)
model_pkl.eval() # Set to evaluation mode

def predict_image_pkl(image):
    # Ensure the image is in RGB format
    if image.mode != 'RGB':
        image = image.convert('RGB')

    # Apply the same transformations as during training
    input_tensor = transform(image)
    input_batch = input_tensor.unsqueeze(0)  # Add a batch dimension

    # Move the input to the same device as the model
    input_batch = input_batch.to(device)
    model_pkl.to(device)

    with torch.no_grad():
        output = model_pkl(input_batch)

    # Get probabilities
    probabilities = F.softmax(output, dim=1)[0]

    # Convert to a dictionary for Gradio output
    confidences = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}

    return confidences

print("Prediction function using model.pkl defined.")

**Reasoning**:
Now that the `predict_image_pkl` function is successfully defined, I will create and launch the Gradio interface using this function. This involves setting up `gr.Interface` with the correct inputs, outputs, and example images, similar to the previous Gradio application.



In [ ]:
import gradio as gr

# Get example image paths from the dataset for Gradio examples
example_images = []

# Take one example from each class for demonstration
# The structure is /kaggle/input/satellite-image-classification/data/{class_name}/{image_name}

# Example for 'cloudy'
example_images.append(os.path.join(path, 'data', 'cloudy', 'cloudy_1.jpg'))
# Example for 'desert'
example_images.append(os.path.join(path, 'data', 'desert', 'desert(1).jpg'))
# Example for 'green_area'
example_images.append(os.path.join(path, 'data', 'green_area', 'Forest_1.jpg'))
# Example for 'water'
example_images.append(os.path.join(path, 'data', 'water', 'SeaLake_1.jpg'))


# Create the Gradio interface using the new prediction function
interface_pkl = gr.Interface(
    fn=predict_image_pkl,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(),
    title="Satellite Image Classifier (from PKL model)",
    description="Upload a satellite image (cloudy, desert, green area, or water) and get its predicted class using the model loaded from model.pkl.",
    examples=example_images
)

# Launch the Gradio app
interface_pkl.launch(debug=True)
print("Gradio interface launched using model.pkl.")

**Reasoning**:
The previous error `AttributeError: 'NoneType' object has no attribute 'mode'` in `predict_image_pkl` indicates that the `image` input to the function was `None`. This can happen if an example image path is invalid or if the Gradio input component passes `None`. I will first modify the `predict_image_pkl` function to explicitly handle `None` input by returning an empty dictionary, making the function more robust. Then, I will correct the `example_images` list in the Gradio interface launch cell to use actual existing image paths from the dataset, as the current ones appear to be incorrect based on the `os.walk` output.



In [ ]:
import gradio as gr
from PIL import Image
import torch.nn.functional as F

# Load the entire model from the .pkl file
model_pkl = torch.load('model.pkl', weights_only=False)
model_pkl.eval() # Set to evaluation mode

def predict_image_pkl(image):
    if image is None:
        return {}

    # Ensure the image is in RGB format
    if image.mode != 'RGB':
        image = image.convert('RGB')

    # Apply the same transformations as during training
    input_tensor = transform(image)
    input_batch = input_tensor.unsqueeze(0)  # Add a batch dimension

    # Move the input to the same device as the model
    input_batch = input_batch.to(device)
    model_pkl.to(device)

    with torch.no_grad():
        output = model_pkl(input_batch)

    # Get probabilities
    probabilities = F.softmax(output, dim=1)[0]

    # Convert to a dictionary for Gradio output
    confidences = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}

    return confidences

print("Prediction function using model.pkl defined.")